# Phase 3, Forward Sequential Feature Selection
---
Forward SFS on 39 shape parameters to find the optimal subset.
Then: nested CV (inner for tuning, outer for evaluation) on selected features.
Metrics: R², MSE, MAE, clinical tolerance ±0.5 and ±1.0 BCS.

In [2]:
import warnings

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold, KFold, GridSearchCV, cross_validate
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.base import BaseEstimator, RegressorMixin
from mord import LogisticAT

warnings.filterwarnings('ignore')

In [3]:
df = pd.read_excel('../COMBINED_HORSES_DATA.xlsx')

beta_cols = [f'beta_{i}' for i in range(0, 39)]
X = df[beta_cols]
y = df['BCS']
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")

Dataset: 208 samples, 39 features


In [4]:
# sklearn-compatible wrapper for mord ordinal regression with non-consecutive label support
class OrdinalWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    # fit ordinal model using a remapped integer encoding of labels
    def fit(self, X, y):
        unique_vals = np.sort(np.unique(y))
        self.label_map_ = {v: i for i, v in enumerate(unique_vals)}
        self.inverse_map_ = {i: v for v, i in self.label_map_.items()}
        y_enc = np.array([self.label_map_[v] for v in y])
        self.model_ = LogisticAT(alpha=self.alpha)
        self.model_.fit(X, y_enc)
        return self

    # decode predicted integer labels back to original BCS values
    def predict(self, X):
        y_enc = self.model_.predict(X)
        return np.array([self.inverse_map_[int(round(v))] for v in y_enc])

## Forward Sequential Feature Selection
---
SFS is wrapped inside a Pipeline (scaler → SFS → OLS) and evaluated
via cross_validate, so scaling and feature selection are both
performed only on the training portion of each fold.

In [5]:
import numpy as np
from sklearn.model_selection import RepeatedKFold, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer


def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)


def clinical_accuracy_05(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred) <= 0.5)


def clinical_accuracy_10(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred) <= 1.0)


cv_strategy = RepeatedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

inner_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

max_features = 15

history_test_mse = []
history_test_r2 = []
history_adj_r2 = []

history_train_mse = []
history_train_r2 = []

history_test_acc_05 = []
history_test_acc_10 = []

best_features_list = []

for i in range(1, max_features + 1):

    sfs_step = SequentialFeatureSelector(
        estimator=LinearRegression(),
        n_features_to_select=i,
        direction='forward',
        cv=inner_cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    eval_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('sfs', sfs_step),
        ('ols', LinearRegression())
    ])

    scores = cross_validate(
        eval_pipe,
        X,
        y,
        cv=cv_strategy,
        scoring={
            'r2': 'r2',
            'neg_mse': 'neg_mean_squared_error',
            'acc_05': make_scorer(clinical_accuracy_05),
            'acc_10': make_scorer(clinical_accuracy_10)
        },
        return_train_score=True,
        n_jobs=-1
    )

    sfs_full = SequentialFeatureSelector(
        estimator=Pipeline([
            ('scaler', StandardScaler()),
            ('ols', LinearRegression())
        ]),
        n_features_to_select=i,
        direction='forward',
        cv=inner_cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    sfs_full.fit(X, y)

    selected = list(X.columns[sfs_full.get_support()])
    best_features_list.append(selected)

    mean_test_mse = -scores['test_neg_mse'].mean()
    mean_test_r2 = scores['test_r2'].mean()

    mean_train_mse = -scores['train_neg_mse'].mean()
    mean_train_r2 = scores['train_r2'].mean()

    mean_acc_05 = scores['test_acc_05'].mean()
    mean_acc_10 = scores['test_acc_10'].mean()

    n = X.shape[0]
    adj_r2 = adjusted_r2(mean_test_r2, n, i)

    history_test_mse.append(mean_test_mse)
    history_test_r2.append(mean_test_r2)
    history_adj_r2.append(adj_r2)

    history_train_mse.append(mean_train_mse)
    history_train_r2.append(mean_train_r2)

    history_test_acc_05.append(mean_acc_05)
    history_test_acc_10.append(mean_acc_10)

    print(
        f"{i:2d} Features | "
        f"Test R²: {mean_test_r2:6.4f} | "
        f"Adj R²: {adj_r2:6.4f} | "
        f"Test MSE: {mean_test_mse:6.4f} | "
        f"±0.5: {mean_acc_05:.1%} | "
        f"±1.0: {mean_acc_10:.1%} | "
        f"{selected}"
    )

optimal_idx = np.argmin(history_test_mse)

optimal_features = best_features_list[optimal_idx]

print(f"\nOptimal: {len(optimal_features)} Features")
print(f"{optimal_features}")

print(f"\nTest MSE: {history_test_mse[optimal_idx]:.4f}")
print(f"Test R²: {history_test_r2[optimal_idx]:.4f}")
print(f"Adjusted R²: {history_adj_r2[optimal_idx]:.4f}")

print(f"±0.5 Accuracy: {history_test_acc_05[optimal_idx]:.1%}")
print(f"±1.0 Accuracy: {history_test_acc_10[optimal_idx]:.1%}")

X_optimal = X[optimal_features]

 1 Features | Test R²: 0.0853 | Adj R²: 0.0809 | Test MSE: 0.6796 | ±0.5: 46.6% | ±1.0: 78.0% | ['beta_1']
 2 Features | Test R²: 0.1365 | Adj R²: 0.1281 | Test MSE: 0.6409 | ±0.5: 47.6% | ±1.0: 79.8% | ['beta_1', 'beta_31']
 3 Features | Test R²: 0.1525 | Adj R²: 0.1400 | Test MSE: 0.6270 | ±0.5: 48.2% | ±1.0: 81.7% | ['beta_1', 'beta_3', 'beta_31']
 4 Features | Test R²: 0.1901 | Adj R²: 0.1741 | Test MSE: 0.5965 | ±0.5: 49.3% | ±1.0: 80.9% | ['beta_1', 'beta_3', 'beta_8', 'beta_31']
 5 Features | Test R²: 0.2225 | Adj R²: 0.2033 | Test MSE: 0.5731 | ±0.5: 50.8% | ±1.0: 79.5% | ['beta_1', 'beta_3', 'beta_7', 'beta_8', 'beta_31']
 6 Features | Test R²: 0.2509 | Adj R²: 0.2285 | Test MSE: 0.5503 | ±0.5: 50.8% | ±1.0: 82.6% | ['beta_1', 'beta_3', 'beta_7', 'beta_8', 'beta_23', 'beta_31']
 7 Features | Test R²: 0.2603 | Adj R²: 0.2344 | Test MSE: 0.5439 | ±0.5: 52.4% | ±1.0: 82.1% | ['beta_1', 'beta_3', 'beta_5', 'beta_7', 'beta_8', 'beta_23', 'beta_31']
 8 Features | Test R²: 0.2467 | A

## Nested CV evaluation on selected features
---
Inner CV (5-fold) tunes hyperparameters per outer fold.
Outer CV (5-fold × 3 repeats) provides unbiased test metrics.
The test fold NEVER influences hyperparameter selection.

In [6]:
# nested CV evaluation: inner fold tunes hyperparameters, outer fold measures test performance
def evaluate_model(model_class, param_grid, X, y, name):
    outer_cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
    inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

    test_r2, test_mse, test_mae = [], [], []
    train_r2, train_mse = [], []
    all_true, all_pred = [], []

    for train_idx, test_idx in outer_cv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        if param_grid:
            gs = GridSearchCV(model_class(), param_grid, cv=inner_cv,
                              scoring='neg_mean_squared_error', n_jobs=-1)
            gs.fit(X_tr_s, y_tr)
            model = gs.best_estimator_
        else:
            model = model_class()
            model.fit(X_tr_s, y_tr)

        tr_pred = model.predict(X_tr_s)
        te_pred = model.predict(X_te_s)

        train_r2.append(r2_score(y_tr, tr_pred))
        train_mse.append(mean_squared_error(y_tr, tr_pred))
        test_r2.append(r2_score(y_te, te_pred))
        test_mse.append(mean_squared_error(y_te, te_pred))
        test_mae.append(mean_absolute_error(y_te, te_pred))
        all_true.extend(y_te)
        all_pred.extend(te_pred)

    errors = np.abs(np.array(all_true) - np.array(all_pred))
    print(f"{name}")
    print(f"  Test R²:  {np.mean(test_r2):.4f}  (Train: {np.mean(train_r2):.4f})")
    print(f"  Test MSE: {np.mean(test_mse):.4f}  (Train: {np.mean(train_mse):.4f})")
    print(f"  Test MAE: {np.mean(test_mae):.4f}")
    print(f"  ±0.5: {np.mean(errors <= 0.5):.1%}  |  ±1.0: {np.mean(errors <= 1.0):.1%}")
    print()


ridge_pg = {'alpha': np.logspace(-2, 3, 30)}
lasso_pg = {'alpha': np.logspace(-4, 1, 30)}
pls_pg = {'n_components': range(1, min(10, X_optimal.shape[1] + 1))}
ordinal_pg = {'alpha': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]}
rf_pg = {'n_estimators': [50, 100, 150],
         'max_depth': [3, 5, 7, 10],
         'min_samples_split': [5, 10],
         'min_samples_leaf': [2, 5]}
mlp_pg = {'hidden_layer_sizes': [(4,), (4, 4), (8,)],
          'activation': ['tanh', 'relu'],
          'alpha': [0.01, 0.1, 1.0, 5.0, 10.0],
          'solver': ['adam', 'lbfgs']}

# nested cv on optimal features
print("Nested CV On Optimal Features:")
print()

evaluate_model(LinearRegression, None, X_optimal, y, "OLS")
evaluate_model(lambda: Ridge(random_state=42), ridge_pg, X_optimal, y, "Ridge")
evaluate_model(lambda: Lasso(random_state=42, max_iter=10_000), lasso_pg, X_optimal, y, "Lasso")
evaluate_model(PLSRegression, pls_pg, X_optimal, y, "PLS")
evaluate_model(lambda: RandomForestRegressor(random_state=42), rf_pg, X_optimal, y, "Random Forest")
evaluate_model(lambda: MLPRegressor(max_iter=2000, random_state=42), mlp_pg, X_optimal, y, "MLP")
evaluate_model(OrdinalWrapper, ordinal_pg, X_optimal, y, "Ordinal Regression")

Nested CV On Optimal Features:

OLS
  Test R²:  0.3915  (Train: 0.4668)
  Test MSE: 0.4486  (Train: 0.4097)
  Test MAE: 0.5231
  ±0.5: 57.2%  |  ±1.0: 85.9%

Ridge
  Test R²:  0.3925  (Train: 0.4650)
  Test MSE: 0.4482  (Train: 0.4111)
  Test MAE: 0.5245
  ±0.5: 57.9%  |  ±1.0: 85.9%

Lasso
  Test R²:  0.3927  (Train: 0.4664)
  Test MSE: 0.4478  (Train: 0.4100)
  Test MAE: 0.5234
  ±0.5: 57.4%  |  ±1.0: 85.7%

PLS
  Test R²:  0.3894  (Train: 0.4653)
  Test MSE: 0.4501  (Train: 0.4108)
  Test MAE: 0.5238
  ±0.5: 57.4%  |  ±1.0: 85.7%

Random Forest
  Test R²:  0.3001  (Train: 0.7204)
  Test MSE: 0.5193  (Train: 0.2140)
  Test MAE: 0.5753
  ±0.5: 51.4%  |  ±1.0: 82.9%

MLP
  Test R²:  0.3849  (Train: 0.4945)
  Test MSE: 0.4541  (Train: 0.3885)
  Test MAE: 0.5276
  ±0.5: 56.2%  |  ±1.0: 85.3%

Ordinal Regression
  Test R²:  0.3203  (Train: 0.3957)
  Test MSE: 0.4998  (Train: 0.4643)
  Test MAE: 0.4518
  ±0.5: 57.2%  |  ±1.0: 97.6%



## VotingRegressor Ensemble
---
Each ensemble member is tuned per outer fold.
Ordinal Regression excluded (integer output).

In [7]:
CONFIGS = [
    ('ridge', lambda: Ridge(random_state=42), ridge_pg),
    ('lasso', lambda: Lasso(random_state=42, max_iter=10_000), lasso_pg),
    ('pls',   PLSRegression, pls_pg),
    ('rf',    lambda: RandomForestRegressor(random_state=42), rf_pg),
    ('mlp',   lambda: MLPRegressor(max_iter=2000, random_state=42), mlp_pg),
]

outer_cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

ens_r2, ens_mse, ens_mae = [], [], []
all_true, all_pred = [], []

for train_idx, test_idx in outer_cv.split(X_optimal):
    X_tr, X_te = X_optimal.iloc[train_idx], X_optimal.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    members = []
    for mem_name, model_func, pg in CONFIGS:
        gs = GridSearchCV(model_func(), pg, cv=inner_cv,
                          scoring='neg_mean_squared_error', n_jobs=-1)
        gs.fit(X_tr_s, y_tr)
        members.append((mem_name, gs.best_estimator_))

    ens = VotingRegressor(estimators=members)
    ens.fit(X_tr_s, y_tr)
    preds = ens.predict(X_te_s)

    ens_r2.append(r2_score(y_te, preds))
    ens_mse.append(mean_squared_error(y_te, preds))
    ens_mae.append(mean_absolute_error(y_te, preds))
    all_true.extend(y_te)
    all_pred.extend(preds)

errors = np.abs(np.array(all_true) - np.array(all_pred))
print("Voting Regressor (Ridge + Lasso + PLS + RF + MLP)")
print(f"  Test R²:  {np.mean(ens_r2):.4f}")
print(f"  Test MSE: {np.mean(ens_mse):.4f}")
print(f"  Test MAE: {np.mean(ens_mae):.4f}")
print(f"  ±0.5: {np.mean(errors <= 0.5):.1%}  |  ±1.0: {np.mean(errors <= 1.0):.1%}")

Voting Regressor (Ridge + Lasso + PLS + RF + MLP)
  Test R²:  0.3981
  Test MSE: 0.4447
  Test MAE: 0.5241
  ±0.5: 57.9%  |  ±1.0: 86.2%
